## Automate Data Extraction & Finalize Raw Dataset

**Objective:**  
The purpose of this phase is to automate the process of collecting military data. Rather than manually gathering statistics for more than 140 countries, a Python script is used to systematically access pages from the Global Firepower index. The script processes a list of country-specific URLs, retrieves important indicators such as manpower figures, defense expenditure, and equipment inventories, and consolidates this information into an initial raw dataset for further processing.

In [3]:
import pandas as pd
import requests
import re
import time
from bs4 import BeautifulSoup

In [4]:
# 1. Get Country List
countries_url = "https://www.globalfirepower.com/countries-listing.php"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(countries_url, headers=headers, timeout=10)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

country_tags = soup.find_all("a", href=True)
ids = []

for a in country_tags:
    href = str(a["href"])
    if "country-military-strength-detail.php?country_id=" in href:
        cid = href.split("country_id=")[1]
        ids.append(cid)

ids = list(set(ids))

# 2. Updated Mapping with Justifications (Vehicles removed)
key_map = {
    "total_population": "Total Population:",  # Baseline for calculating per-capita metrics and overall national scale.
    "gdp": "Purchasing Power Parity:",        # Economic capacity and denominator for the Budget-to-GDP KPI.
    "defense_budget": "Defense Budget:",      # Financial strength and absolute military spending power.
    "total_manpower": "Available Manpower",   # Overall potential human resource pool for wartime mobilization.
    "active_personnel": "Active Personnel",   # The current, immediate standing military force size.
    "reserve_personnel": "Reserve Personnel", # Backup military manpower available for extended conflicts.
    "air_force_personnel": "Air Force Personnel*", # Specialized human capital dedicated to air dominance.
    "army_personnel": "Army Personnel*",      # Core ground force human capital for territorial control.
    "navy_personnel": "Navy Personnel*",      # Specialized human capital for maritime operations.
    "total_aircraft": "Aircraft Total:",      # High-level indicator of overall air power capability.
    "attack_aircraft": "Attack Types:",       # Measures offensive air-to-ground strike capability.
    "fighter_aircraft": "Fighters:",          # Measures air-to-air combat and air superiority strength.
    "transport_aircraft": "Transports (Fixed-Wing):", # Indicates logistics scale and troop deployment range.
    "helicopters": "Helicopters:",            # Tactical mobility and rapid ground support capability.
    "special_mission_aircraft": "Special-Mission:", # Measures reconnaissance, intelligence, and electronic warfare capacity.
    "trainer_aircraft": "Trainers:",          # Indicates long-term capacity to generate new pilots.
    "attack_helicopters": "Attack Helicopters:", # Specialized close air support for ground troops/tank hunting.
    "tanker_aircraft": "Tanker Fleet:",       # Force multiplier indicating extended range and global reach.
    "naval_assets": "Total Assets:",          # High-level indicator of overall maritime and coastal power.
    "aircraft_carriers": "Aircraft Carriers:",# The ultimate measure of global power projection and mobile airbases.
    "helicopter_carriers": "Helicopter Carriers:", # Capability for amphibious assaults and marine deployments.
    "destroyers": "Destroyers:",              # Heavy, multi-role blue-water surface combatants.
    "frigates": "Frigates:",                  # Medium surface combatants for escort and anti-submarine warfare.
    "corvettes": "Corvettes:",                # Light, fast vessels primarily for coastal and regional defense.
    "submarines": "Submarines:",              # Crucial metric for stealth strike capability and naval deterrence.
    "patrol_vessel": "Patrol Vessels:",       # Indicates capacity for border security and exclusive economic zone (EEZ) patrol.
    "mine_warfare": "Mine Warfare:",          # Capacity for area denial and keeping vital shipping lanes open.
    "tanks": "Tanks:",                        # Core metric for armored ground assault and holding territory.
    "self_propelled_artillery": "Self-Propelled Artillery:", # Highly mobile, survivable indirect fire support.
    "towed_artillery": "Towed Artillery:",    # Traditional, static indirect fire support for ground forces.
    "rocket_artillery": "MLRS (Rocket Artillery):", # Capability for rapid, devastating area-of-effect bombardments.
    "external_debt": "External Debt:"         # Economic liability that could cripple long-term war sustainment.
}

# 3. Automation Function
def automation(ids, key_map):
    base_url = "https://www.globalfirepower.com/country-military-strength-detail.php?country_id="
    all_countries = {}

    for cid in ids:
        try:
            url = base_url + cid
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            country_name = cid.replace("-", " ").title()

            # Extract Power Index
            power_index = None
            info_block = soup.select_one("span.textNormal.textDkGray")
            if info_block:
                match = re.search(r"\d+\.\d+", info_block.text)
                if match:
                    power_index = match.group()

            # Extract Specifications
            specs_containers = soup.find_all("div", class_="specsGenContainers")
            data_found = {}

            for container in specs_containers:
                tag = container.select_one("span.textLarge.textBold.textShadow")
                value_tags = container.select("span.textWhite.textShadow")

                if tag and value_tags:
                    name = tag.text.strip()
                    # We take the last value in the list because GFP often puts
                    # the rank/label first and the actual number last
                    values = [v.text.strip() for v in value_tags]
                    data_found[name] = values[-1]

            # Map the results
            result = {}
            for new_key, old_key in key_map.items():
                if old_key in data_found:
                    result[new_key] = data_found[old_key]
                else:
                    result[new_key] = None

            result["power_index"] = power_index
            all_countries[country_name] = result

            print(f"Scraped: {country_name}")
            time.sleep(1)

        except Exception as e:
            print(f"Error scraping {cid}: {e}")

    return all_countries

# 4. Execution
all_countries_data = automation(ids, key_map)

df = pd.DataFrame.from_dict(all_countries_data, orient="index")
df.reset_index(inplace=True)
df.rename(columns={"index": "country"}, inplace=True)

df.to_csv("military_raw_data.csv", index=False)
print("Done. File saved as military_raw_data.csv")


Scraped: Burkina Faso
Scraped: Turkey
Scraped: Spain
Scraped: Finland
Scraped: Portugal
Scraped: Republic Of The Congo
Scraped: Panama
Scraped: Latvia
Scraped: Ivory Coast
Scraped: Iceland
Scraped: Libya
Scraped: Tunisia
Scraped: Saudi Arabia
Scraped: Slovenia
Scraped: Egypt
Scraped: Uruguay
Scraped: Israel
Scraped: Cameroon
Scraped: Laos
Scraped: Montenegro
Scraped: Central African Republic
Scraped: Algeria
Scraped: Bahrain
Scraped: Nepal
Scraped: Argentina
Scraped: Bolivia
Scraped: Slovakia
Scraped: Armenia
Scraped: Gabon
Scraped: Moldova
Scraped: United Kingdom
Scraped: Bosnia And Herzegovina
Scraped: Bulgaria
Scraped: Canada
Scraped: Ireland
Scraped: Croatia
Scraped: Georgia
Scraped: Qatar
Scraped: Iran
Scraped: Austria
Scraped: Benin
Scraped: Angola
Scraped: Paraguay
Scraped: Belgium
Scraped: Mali
Scraped: Norway
Scraped: Lebanon
Scraped: Ukraine
Scraped: Senegal
Scraped: Turkmenistan
Scraped: Morocco
Scraped: United Arab Emirates
Scraped: Kuwait
Scraped: Bhutan
Scraped: North Mac

## Data Cleaning & Standardization

**Objective:**  
The aim of this phase is to prepare the collected raw data so it can be used for proper analysis and calculations. Since computers cannot perform mathematical operations on text values that include symbols (for example "$500" or "50%"), the dataset must be cleaned and formatted. In this step, unnecessary characters and formatting symbols are removed, column names are standardized for uniformity, and missing or incomplete values are addressed. As a result, the dataset becomes structured and contains properly formatted numeric values that are ready for further analysis.

In [5]:
import pandas as pd
import numpy as np
import sys
import re

In [6]:
# Force UTF-8 output just in case
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

# --- CONFIGURATION ---
INPUT_FILE = "military_raw_data.csv"
OUTPUT_FILE = "military_cleaned.csv"

# --- STEP 1: LOAD RAW DATA ---
print(f"Step 1: Loading raw data from '{INPUT_FILE}'...")
try:
    df = pd.read_csv(INPUT_FILE)
    print("   -> Raw data loaded successfully.")
    print(f"   -> Initial Shape: {df.shape}")
except FileNotFoundError:
    raise Exception(f"ERROR: Could not find '{INPUT_FILE}'.")

# --- STEP 4: STANDARDIZE COLUMN NAMES ---
print("\nStep 4: Standardizing column names (snake_case)...")
df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
print(f"   -> Columns standardized: {list(df.columns[:5])}...")

# --- STEP 2 & 3: CLEAN NUMERIC COLUMNS ---
print("\nStep 2 & 3: Cleaning numeric columns...")

def clean_numeric_value(x):
    if pd.isna(x): return np.nan
    x = str(x).strip().replace(',', '').replace('$', '').replace('%', '').replace('+', '')
    if x.lower() in ['n/a', 'nan', 'null', '', 'none']: return np.nan

    # Use RegEx to find the first sequence of numbers (ignores words like "Stock:")
    match = re.search(r'([\d.]+)', x)
    if match:
        return float(match.group(1))
    return np.nan

cols_to_clean = [col for col in df.columns if 'country' not in col]
for col in cols_to_clean:
    df[col] = df[col].apply(clean_numeric_value)

print(f"   -> Cleaned {len(cols_to_clean)} numeric columns.")

# --- STEP 5: HANDLING MISSING VALUES ---
print("\nStep 5: Handling missing values...")
null_counts = df.isnull().sum()
missing_cols = null_counts[null_counts > 0]
if not missing_cols.empty:
    print(f"   -> Found missing values in:\n{missing_cols}")
else:
    print("   -> No missing values found initially.")

df = df.fillna(0)
print("   -> Applied Rule: Filled missing values with 0.")

# --- STEP 6: VALIDATE CLEANED DATA ---
print("\nStep 6: Validating cleaned data...")

row_count = len(df)
if row_count < 140:
    print(f"   WARNING: Row count is {row_count}. Check scraping completeness.")
else:
    print(f"   [OK] Row count check passed: {row_count} countries.")

non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
if len(non_numeric_cols) > 1:
    print(f"   WARNING: Unexpected non-numeric columns: {non_numeric_cols}")
else:
    print("   [OK] Data type check passed (Metrics are numeric).")

if df.isnull().sum().sum() == 0:
    print("   [OK] Null value check passed (0% missing).")
else:
    print("   WARNING: There are still null values remaining.")

# --- STEP 7: EXPORT ---
print(f"\nStep 7: Exporting to '{OUTPUT_FILE}'...")
df.to_csv(OUTPUT_FILE, index=False)
print(f"[SUCCESS] Cleaned dataset saved as '{OUTPUT_FILE}'.")
print("   -> Ready for KPI Engineering (Task 7).")


Step 1: Loading raw data from 'military_raw_data.csv'...
   -> Raw data loaded successfully.
   -> Initial Shape: (145, 34)

Step 4: Standardizing column names (snake_case)...
   -> Columns standardized: ['country', 'total_population', 'gdp', 'defense_budget', 'total_manpower']...

Step 2 & 3: Cleaning numeric columns...
   -> Cleaned 33 numeric columns.

Step 5: Handling missing values...
   -> Found missing values in:
naval_assets           32
aircraft_carriers      32
helicopter_carriers    32
destroyers             32
frigates               32
corvettes              32
submarines             32
patrol_vessel          32
mine_warfare           32
dtype: int64
   -> Applied Rule: Filled missing values with 0.

Step 6: Validating cleaned data...
   [OK] Row count check passed: 145 countries.
   [OK] Data type check passed (Metrics are numeric).
   [OK] Null value check passed (0% missing).

Step 7: Exporting to 'military_cleaned.csv'...
[SUCCESS] Cleaned dataset saved as 'military_cle

**Takeaway:**  
A total of 30 numeric columns were cleaned and standardized. During the process, 32 missing values (approximately 22%) were identified within the Naval Asset fields. These missing entries were primarily due to landlocked countries that do not possess naval forces, and they were handled by imputing the values as zero to maintain dataset consistency.